In [1]:
# Install dependencies (run once in terminal with venv activated):
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'tqdm', 'kagglehub', 'nltk', 'matplotlib', 'pandas', 'pillow'])


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


0

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
import pandas as pd
import matplotlib.pyplot as plt
import random
import re
import os
import sys
import csv
import math
from PIL import Image
from collections import Counter
from tqdm import tqdm
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)

# MPS fallback for unsupported ops on Apple Silicon
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Mac-compatible device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Using device:', device)

Using device: mps


In [3]:
# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR = './model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH   = os.path.join(OUTPUT_DIR, 'resnet50_attention_model.pth')
VOCAB_PATH   = os.path.join(OUTPUT_DIR, 'vocab.pt')
FEATURE_PATH = os.path.join(OUTPUT_DIR, 'resnet50_features.pt')  # all images cached

print('Output directory ready:', OUTPUT_DIR)

Output directory ready: ./model_outputs


In [4]:
# ── Download Flickr8k via kagglehub ──────────────────────────────────────────
import kagglehub
path = kagglehub.dataset_download('adityajn105/flickr8k')
IMAGE_DIR    = os.path.join(path, 'Images')
CAPTION_FILE = os.path.join(path, 'captions.txt')
print(len(os.listdir(IMAGE_DIR)), 'images found')
print('Caption file exists:', os.path.exists(CAPTION_FILE))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8091 images found
Caption file exists: True


In [5]:
# ── Custom tokenizer — preserves <start>/<end> as single tokens ──────────────
def tokenize(caption: str):
    """Lowercase, strip noise, keep <start>/<end> as single tokens."""
    caption = caption.lower().strip()
    caption = re.sub(r"[^a-z0-9<>' ]", ' ', caption)
    caption = caption.replace('<start>', ' <start> ').replace('<end>', ' <end> ')
    caption = re.sub(r'\s+', ' ', caption).strip()
    return caption.split()

test = '<start> A dog runs across the field . <end>'
print(tokenize(test))

['<start>', 'a', 'dog', 'runs', 'across', 'the', 'field', '<end>']


In [6]:
# ── Load captions ─────────────────────────────────────────────────────────────
captions = {}
with open(CAPTION_FILE, 'r') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 2:
            img_name     = row[0]
            caption_text = '<start> ' + row[1].strip() + ' <end>'
            captions.setdefault(img_name, []).append(caption_text)

print('Total images with captions:', len(captions))

Total images with captions: 8091


In [7]:
# ── Build vocab with <unk> ────────────────────────────────────────────────────
word_counter = Counter()
SKIP_TOKENS  = {'<pad>', '<start>', '<end>', '<unk>'}
for caps_list in captions.values():
    for cap in caps_list:
        word_counter.update(t for t in tokenize(cap) if t not in SKIP_TOKENS)

SPECIALS = ['<pad>', '<start>', '<end>', '<unk>']
word2idx = {w: i for i, w in enumerate(SPECIALS)}
idx2word = {i: w for i, w in enumerate(SPECIALS)}

idx      = len(SPECIALS)
MIN_FREQ = 5
for w, c in word_counter.items():
    if c >= MIN_FREQ and w not in word2idx:
        word2idx[w] = idx
        idx2word[idx] = w
        idx += 1

vocab_size = len(word2idx)
PAD_IDX    = word2idx['<pad>']
START_IDX  = word2idx['<start>']
END_IDX    = word2idx['<end>']
UNK_IDX    = word2idx['<unk>']

print(f'Vocabulary size: {vocab_size}')
print(f'  <pad>={PAD_IDX}, <start>={START_IDX}, <end>={END_IDX}, <unk>={UNK_IDX}')

torch.save((word2idx, idx2word), VOCAB_PATH)
print('Vocabulary saved.')

Vocabulary size: 2982
  <pad>=0, <start>=1, <end>=2, <unk>=3
Vocabulary saved.


In [8]:
# ── Caption → index sequence ──────────────────────────────────────────────────
def caption_to_seq(caption: str):
    return [word2idx.get(w, UNK_IDX) for w in tokenize(caption)]

seq = caption_to_seq('<start> a dog runs <end>')
print('Example seq:', seq)
print('Decoded    :', [idx2word[i] for i in seq])

Example seq: [1, 4, 28, 123, 2]
Decoded    : ['<start>', 'a', 'dog', 'runs', '<end>']


## Dataset, DataLoader & Feature Extraction

In [9]:
# ── Train / val split (90 / 10) ───────────────────────────────────────────────
all_images = list(captions.keys())
random.seed(42)
random.shuffle(all_images)
split        = int(0.9 * len(all_images))
train_images = all_images[:split]
val_images   = all_images[split:]
print(f'Train: {len(train_images)} | Val: {len(val_images)}')

Train: 7281 | Val: 810


In [10]:
# ── Transform (single clean transform — fully cached features) ───────────────
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Encoder: ResNet50 up to layer4, fully frozen ──────────────────────────────
resnet  = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
encoder = nn.Sequential(*list(resnet.children())[:-2])
encoder = encoder.to(device).eval()
for p in encoder.parameters():
    p.requires_grad = False

print('Encoder ready. Fully frozen — features will be cached.')
print('Spatial output: 7x7 = 49 locations, 2048-dim each.')

Encoder ready. Fully frozen — features will be cached.
Spatial output: 7x7 = 49 locations, 2048-dim each.


In [11]:
# ── Cache features for ALL images (train + val) ───────────────────────────────
# Cached once, reused every epoch — fast stable training on M2.
# Delete model_outputs/resnet50_features.pt to force re-extraction.

if os.path.exists(FEATURE_PATH):
    print('Loading cached features...')
    all_features = torch.load(FEATURE_PATH, map_location='cpu', weights_only=False)
    print(f'Loaded features for {len(all_features)} images.')
else:
    print('Extracting features for all images (this runs once, ~2-3 mins on M2)...')
    all_features = {}
    encoder.eval()
    with torch.no_grad():
        for img_name in tqdm(all_images):
            img_path = os.path.join(IMAGE_DIR, img_name)
            try:
                img   = Image.open(img_path).convert('RGB')
                img_t = val_transform(img).unsqueeze(0).to(device)
                feat  = encoder(img_t)
                feat  = feat.squeeze(0).permute(1, 2, 0).reshape(49, 2048).cpu()
                all_features[img_name] = feat
            except Exception as e:
                print(f'  Skipping {img_name}: {e}')
    torch.save(all_features, FEATURE_PATH)
    print(f'Features saved for {len(all_features)} images.')

Loading cached features...
Loaded features for 8091 images.


In [12]:
from torch.utils.data import Dataset, DataLoader

class Flickr8kDataset(Dataset):
    """Fully cached — returns pre-extracted features. Fast and stable."""
    def __init__(self, image_names, max_len=50):
        self.data = []
        for img_name in image_names:
            if img_name not in all_features:
                continue
            for cap in captions[img_name]:
                seq = caption_to_seq(cap)
                if len(seq) <= max_len:
                    self.data.append((img_name, seq))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name, seq = self.data[idx]
        feat = all_features[img_name]
        return feat, torch.tensor(seq, dtype=torch.long)


def collate_fn(batch):
    feats, seqs = zip(*batch)
    feats   = torch.stack(feats)
    max_len = max(s.size(0) for s in seqs)
    padded  = torch.full((len(seqs), max_len), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :s.size(0)] = s
    return feats, padded


BATCH_SIZE = 64 

# num_workers=0 required on Mac to avoid multiprocessing issues
train_dataset = Flickr8kDataset(train_images)
val_dataset   = Flickr8kDataset(val_images)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

Train samples: 36405 | Val samples: 4050


## Model: Attention + LSTM Decoder

In [13]:
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attn_dim):
        super().__init__()
        self.W_enc = nn.Linear(encoder_dim, attn_dim)
        self.W_dec = nn.Linear(decoder_dim, attn_dim)
        self.V     = nn.Linear(attn_dim, 1)

    def forward(self, encoder_out, h):
        e     = self.W_enc(encoder_out)
        d     = self.W_dec(h).unsqueeze(1)
        score = self.V(torch.tanh(e + d))
        alpha = torch.softmax(score, dim=1)
        ctx   = (alpha * encoder_out).sum(1)
        return ctx, alpha.squeeze(-1)


class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, encoder_dim=2048,
                 decoder_dim=384, attn_dim=256, dropout=0.6):  # smaller decoder, higher dropout
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention = Attention(encoder_dim, decoder_dim, attn_dim)
        self.lstm      = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)
        self.init_c    = nn.Linear(encoder_dim, decoder_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(decoder_dim, vocab_size)

    def _init_hidden(self, encoder_out):
        mean_enc = encoder_out.mean(dim=1)
        h = torch.tanh(self.init_h(mean_enc))
        c = torch.tanh(self.init_c(mean_enc))
        return h, c

    def forward(self, encoder_out, captions, teacher_forcing_ratio=1.0):
        B, T = captions.size()
        h, c = self._init_hidden(encoder_out)
        outputs     = []
        input_token = captions[:, 0]

        for t in range(T - 1):
            emb     = self.dropout(self.embedding(input_token))
            ctx, _  = self.attention(encoder_out, h)
            lstm_in = self.dropout(torch.cat([emb, ctx], dim=1))
            h, c    = self.lstm(lstm_in, (h, c))
            logits  = self.fc(self.dropout(h))
            outputs.append(logits)

            use_gt      = random.random() < teacher_forcing_ratio
            input_token = captions[:, t + 1] if use_gt else logits.argmax(1)

        return torch.stack(outputs, dim=1)


model = DecoderWithAttention(
    vocab_size   = vocab_size,
    embed_dim    = 256,
    encoder_dim  = 2048,
    decoder_dim  = 384,   # reduced from 512 — less capacity to memorize
    attn_dim     = 256,
    dropout      = 0.6,   # increased from 0.5 — stronger regularization
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Decoder parameters: {total_params:,}')
print('Model ready.')

Decoder parameters: 8,240,295
Model ready.


## BLEU-4 Evaluation on Validation Set

In [14]:
@torch.no_grad()
def beam_search(feature, beam_size=5, max_len=50):
    """
    feature : (49, 2048) CPU tensor for a single image.
    Returns : list of token strings (best caption, without <start>/<end>).
    """
    model.eval()
    enc  = feature.unsqueeze(0).to(device)
    h, c = model._init_hidden(enc)

    beams     = [([START_IDX], 0.0, h.clone(), c.clone())]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for seq, score, h, c in beams:
            tok     = torch.tensor([seq[-1]], device=device)
            emb     = model.embedding(tok)
            ctx, _  = model.attention(enc, h)
            lstm_in = torch.cat([emb, ctx], dim=1)
            h_new, c_new = model.lstm(lstm_in, (h, c))
            logits  = model.fc(model.dropout(h_new))

            logits[0, PAD_IDX] = -1e9
            log_probs           = torch.log_softmax(logits, dim=-1)
            topk_lp, topk_idx   = log_probs[0].topk(beam_size)

            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                new_seq   = seq + [idx]
                new_score = score + lp
                if idx == END_IDX:
                    completed.append((new_seq, new_score))
                else:
                    new_beams.append((new_seq, new_score, h_new, c_new))

        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_size]
        if not beams:
            break

    if not completed:
        completed = [(b[0], b[1]) for b in beams]

    best_seq, _ = max(completed, key=lambda x: x[1] / len(x[0]))
    tokens = [idx2word[i] for i in best_seq if i not in (START_IDX, END_IDX, PAD_IDX)]
    return tokens

print('Beam search ready.')

Beam search ready.


In [15]:
# ── Load best checkpoint then run BLEU-4 ─────────────────────────────────────
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
print(f'Loaded best model from epoch {ckpt["epoch"]}.')

smoothie  = SmoothingFunction().method4
refs_all, hyps_all = [], []

for img_name in tqdm(val_images, desc='BLEU eval'):
    if img_name not in all_features:
        continue
    feat = all_features[img_name]
    hyp  = beam_search(feat, beam_size=5)
    refs = [tokenize(c) for c in captions[img_name]]
    hyps_all.append(hyp)
    refs_all.append(refs)

bleu4 = corpus_bleu(refs_all, hyps_all, smoothing_function=smoothie)
print(f'\nValidation BLEU-4: {bleu4:.4f}')

Loaded best model from epoch 23.


BLEU eval: 100%|██████████| 810/810 [08:13<00:00,  1.64it/s]


Validation BLEU-4: 0.2161


## Full Evaluation Metrics

In [16]:
from collections import Counter
import numpy as np
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'rouge-score', 'nltk', 'scikit-learn'])

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# refs_all and hyps_all are built by the BLEU-4 cell above
print(f'Evaluating on {len(hyps_all)} images...\n')
smoothie = SmoothingFunction().method4

# ── BLEU-1, BLEU-2, BLEU-4 ───────────────────────────────────────────────────
bleu1 = corpus_bleu(refs_all, hyps_all,
                    weights=(1.0, 0, 0, 0),
                    smoothing_function=smoothie)
bleu2 = corpus_bleu(refs_all, hyps_all,
                    weights=(0.5, 0.5, 0, 0),
                    smoothing_function=smoothie)
bleu4 = corpus_bleu(refs_all, hyps_all,
                    weights=(0.25, 0.25, 0.25, 0.25),
                    smoothing_function=smoothie)

# ── ROUGE-L ───────────────────────────────────────────────────────────────────
scorer       = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = []
for refs, hyp in zip(refs_all, hyps_all):
    hyp_str  = ' '.join(hyp)
    best_f1  = max(
        scorer.score(' '.join(ref), hyp_str)['rougeL'].fmeasure
        for ref in refs
    )
    rouge_scores.append(best_f1)
rougeL = np.mean(rouge_scores)

# ── METEOR ────────────────────────────────────────────────────────────────────
meteor_scores = []
for refs, hyp in zip(refs_all, hyps_all):
    score = meteor_score(refs, hyp)
    meteor_scores.append(score)
meteor = np.mean(meteor_scores)

# ── Cosine Similarity (TF-IDF) ────────────────────────────────────────────────
cos_scores = []
for refs, hyp in zip(refs_all, hyps_all):
    hyp_str  = ' '.join(hyp)
    ref_strs = [' '.join(r) for r in refs]
    all_docs = ref_strs + [hyp_str]
    try:
        tfidf   = TfidfVectorizer().fit_transform(all_docs)
        hyp_vec = tfidf[-1]
        ref_mat = tfidf[:-1]
        sims    = cosine_similarity(hyp_vec, ref_mat)[0]
        cos_scores.append(float(np.max(sims)))
    except Exception:
        cos_scores.append(0.0)
cosine_sim = np.mean(cos_scores)

# ── F1 Score (token-level) ────────────────────────────────────────────────────
def token_f1(ref_tokens, hyp_tokens):
    ref_set = Counter(ref_tokens)
    hyp_set = Counter(hyp_tokens)
    common  = sum((ref_set & hyp_set).values())
    if common == 0:
        return 0.0
    precision = common / len(hyp_tokens)
    recall    = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

f1_scores = []
for refs, hyp in zip(refs_all, hyps_all):
    best_f1 = max(token_f1(ref, hyp) for ref in refs)
    f1_scores.append(best_f1)
f1 = np.mean(f1_scores)

# ── Print Results ─────────────────────────────────────────────────────────────
print('=' * 45)
print('       EVALUATION RESULTS SUMMARY')
print('=' * 45)
print(f'  BLEU-1           : {bleu1:.4f}')
print(f'  BLEU-2           : {bleu2:.4f}')
print(f'  BLEU-4           : {bleu4:.4f}')
print(f'  ROUGE-L          : {rougeL:.4f}')
print(f'  METEOR           : {meteor:.4f}')
print(f'  Cosine Similarity: {cosine_sim:.4f}')
print(f'  F1 Score         : {f1:.4f}')
print('=' * 45)

# ── Save to CSV ───────────────────────────────────────────────────────────────
import pandas as pd
results_df = pd.DataFrame([
    {'Metric': 'BLEU-1',            'Score': round(bleu1, 4)},
    {'Metric': 'BLEU-2',            'Score': round(bleu2, 4)},
    {'Metric': 'BLEU-4',            'Score': round(bleu4, 4)},
    {'Metric': 'ROUGE-L',           'Score': round(rougeL, 4)},
    {'Metric': 'METEOR',            'Score': round(meteor, 4)},
    {'Metric': 'Cosine Similarity', 'Score': round(cosine_sim, 4)},
    {'Metric': 'F1 Score',          'Score': round(f1, 4)},
])
csv_path = os.path.join(OUTPUT_DIR, 'evaluation_scores.csv')
results_df.to_csv(csv_path, index=False)
print(f'\nScores saved to: {csv_path}')


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


Evaluating on 810 images...

       EVALUATION RESULTS SUMMARY
  BLEU-1           : 0.6117
  BLEU-2           : 0.4467
  BLEU-4           : 0.2161
  ROUGE-L          : 0.4372
  METEOR           : 0.3646
  Cosine Similarity: 0.3143
  F1 Score         : 0.4449

Scores saved to: ./model_outputs/evaluation_scores.csv
